# ElementalTask-RML
## Notebook 05 — Cross-Model Stability

This notebook compares emergence ordering and function-vector geometry across multiple model families.

Goal:
- compare emergence trajectories,
- measure cross-model rank stability,
- compare forecasting behavior,
- evaluate whether capability emergence monitoring generalizes across training systems.

Emergence ≠ magic. Monitor constraints. 📐

## Notebook role in Phase I

- Notebook 01: emergence ordering
- Notebook 02: function-vector geometry
- Notebook 03: drift / instability flags
- Notebook 04: compositional forecasting
- Notebook 05: cross-model stability

Core question:

> Do emergence ordering and function-vector geometry remain stable across model families?

In [ ]:
from pathlib import Path
import os
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import spearmanr
except Exception:
    spearmanr = None

warnings.filterwarnings("ignore")

In [ ]:
# ---------------------------------------------------------------------
# Path detection
# ---------------------------------------------------------------------
def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "dataset").exists() or (p / "notebooks_rml").exists() or (p / "function_vecs").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
RML_DIR = REPO_ROOT / "notebooks_rml"
FIG_DIR = RML_DIR / "figures"
RES_DIR = RML_DIR / "results"
DOC_DIR = RML_DIR / "docs"

for d in [RML_DIR, FIG_DIR, RES_DIR, DOC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_DIR:", RML_DIR)
print("FIG_DIR:", FIG_DIR)
print("RES_DIR:", RES_DIR)

## 1. Load multi-model results

Expected structure for real results:

| model | checkpoint | task | score |
|---|---:|---|---:|

If real multi-model CSVs are unavailable, this notebook uses a deterministic synthetic fallback to keep the monitoring pipeline reproducible.

In [ ]:
def detect_multimodel_csvs():
    search_dirs = [
        REPO_ROOT / "results",
        REPO_ROOT / "output",
        RML_DIR / "results",
        RES_DIR,
    ]
    found = []
    for d in search_dirs:
        if d.exists():
            for p in d.glob("*.csv"):
                try:
                    cols = set(pd.read_csv(p, nrows=5).columns.str.lower())
                    if {"model", "checkpoint", "task"}.issubset(cols) and (("score" in cols) or ("accuracy" in cols) or ("acc" in cols)):
                        found.append(p)
                except Exception:
                    pass
    return found

csvs = detect_multimodel_csvs()
csvs

In [ ]:
def load_real_multimodel_results(csvs):
    frames = []
    for p in csvs:
        df = pd.read_csv(p)
        # normalize columns
        df.columns = [c.lower() for c in df.columns]
        if "accuracy" in df.columns and "score" not in df.columns:
            df = df.rename(columns={"accuracy": "score"})
        if "acc" in df.columns and "score" not in df.columns:
            df = df.rename(columns={"acc": "score"})
        frames.append(df[["model", "checkpoint", "task", "score"]].copy())
    if not frames:
        return None
    out = pd.concat(frames, ignore_index=True)
    out["checkpoint"] = pd.to_numeric(out["checkpoint"], errors="coerce")
    out["score"] = pd.to_numeric(out["score"], errors="coerce")
    out = out.dropna(subset=["model", "checkpoint", "task", "score"])
    return out

real_df = load_real_multimodel_results(csvs)
real_df.head() if real_df is not None else None

## 2. Synthetic fallback

This fallback is not a claim about any specific model. It is a reproducible scaffold that mirrors the expected shape of ElementalTask checkpoint outputs.

Models:
- `OLMo-like`
- `OLMo2-like`
- `LLM360-like`

Tasks:
- atomic/simple tasks
- math
- compositional tasks

In [ ]:
def logistic_curve(checkpoints, midpoint, steepness=0.00012, floor=0.0, ceiling=1.0):
    x = np.asarray(checkpoints, dtype=float)
    y = floor + (ceiling - floor) / (1.0 + np.exp(-steepness * (x - midpoint)))
    return y

def build_synthetic_multimodel(seed=42):
    rng = np.random.default_rng(seed)
    checkpoints = np.array([1000, 3000, 5000, 10000, 20000, 50000, 100000])

    base_midpoints = {
        "simple:copying": 2500,
        "simple:uppercase": 6500,
        "simple:first_letter": 12000,
        "math:arithmetic": 22000,
        "compositional:copy_then_uppercase": 45000,
        "compositional:first_letter_then_uppercase": 65000,
    }

    model_params = {
        "OLMo-like": {"scale": 1.00, "jitter": 2000, "noise": 0.035},
        "OLMo2-like": {"scale": 0.88, "jitter": 2500, "noise": 0.030},
        "LLM360-like": {"scale": 1.12, "jitter": 3500, "noise": 0.045},
    }

    rows = []
    for model, params in model_params.items():
        for task, mid in base_midpoints.items():
            model_mid = mid * params["scale"] + rng.normal(0, params["jitter"])
            y = logistic_curve(checkpoints, model_mid, steepness=0.00011)
            y = np.clip(y + rng.normal(0, params["noise"], size=len(checkpoints)), 0, 1)
            # ensure later checkpoint tends to be strongest
            y = np.maximum.accumulate(y * 0.88 + np.linspace(0, 0.12, len(y)))
            for ckpt, score in zip(checkpoints, y):
                rows.append({
                    "model": model,
                    "checkpoint": int(ckpt),
                    "task": task,
                    "score": float(np.clip(score, 0, 1)),
                })
    return pd.DataFrame(rows)

df = real_df if real_df is not None and len(real_df) else build_synthetic_multimodel()
df.to_csv(RES_DIR / "05_multimodel_input.csv", index=False)
df.head(12)

## 3. Compute emergence tables per model

For each model and task, compute the first checkpoint where the score crosses the emergence threshold.

In [ ]:
THRESHOLD = 0.50

def compute_emergence_table(df, threshold=THRESHOLD):
    rows = []
    for (model, task), g in df.sort_values("checkpoint").groupby(["model", "task"]):
        crossed = g[g["score"] >= threshold]
        if len(crossed):
            emergence_ckpt = int(crossed["checkpoint"].iloc[0])
            final_score = float(g.sort_values("checkpoint")["score"].iloc[-1])
            status = "emerged"
        else:
            emergence_ckpt = np.nan
            final_score = float(g.sort_values("checkpoint")["score"].iloc[-1])
            status = "not_emerged"
        rows.append({
            "model": model,
            "task": task,
            "emergence_checkpoint": emergence_ckpt,
            "final_score": final_score,
            "status": status,
        })
    out = pd.DataFrame(rows)
    out["emergence_rank"] = out.groupby("model")["emergence_checkpoint"].rank(method="dense", na_option="bottom")
    return out.sort_values(["model", "emergence_rank", "task"])

emergence = compute_emergence_table(df)
emergence.to_csv(RES_DIR / "05_cross_model_emergence.csv", index=False)
emergence

## 4. Cross-model rank correlation

This metric compares task-emergence ordering between model families.

A high Spearman correlation means models tend to develop task capabilities in similar order.

In [ ]:
def cross_model_rank_correlation(emergence):
    models = sorted(emergence["model"].unique())
    tasks = sorted(emergence["task"].unique())
    rank_matrix = emergence.pivot(index="task", columns="model", values="emergence_rank").reindex(tasks)

    corr = pd.DataFrame(index=models, columns=models, dtype=float)
    for a in models:
        for b in models:
            va = rank_matrix[a]
            vb = rank_matrix[b]
            mask = va.notna() & vb.notna()
            if mask.sum() < 2:
                val = np.nan
            elif spearmanr is not None:
                val = spearmanr(va[mask], vb[mask]).correlation
            else:
                val = va[mask].rank().corr(vb[mask].rank(), method="pearson")
            corr.loc[a, b] = val
    return rank_matrix, corr

rank_matrix, rank_corr = cross_model_rank_correlation(emergence)
rank_corr.to_csv(RES_DIR / "05_cross_model_rank_correlation.csv")
rank_corr

In [ ]:
plt.figure(figsize=(8, 6))
im = plt.imshow(rank_corr.astype(float), vmin=-1, vmax=1)
plt.colorbar(im, label="Spearman rank correlation")
plt.xticks(range(len(rank_corr.columns)), rank_corr.columns, rotation=30, ha="right")
plt.yticks(range(len(rank_corr.index)), rank_corr.index)
plt.title("ElementalTask-RML: Cross-Model Emergence Rank Correlation")

for i in range(len(rank_corr.index)):
    for j in range(len(rank_corr.columns)):
        val = rank_corr.iloc[i, j]
        if pd.notna(val):
            plt.text(j, i, f"{val:.2f}", ha="center", va="center")

plt.tight_layout()
plt.savefig(FIG_DIR / "05_cross_model_rank_correlation.png", dpi=180)
plt.show()

## 5. Cross-model emergence trajectories

Overlay task trajectories by model to see whether developmental patterns share structure across checkpoint schedules.

In [ ]:
tasks = sorted(df["task"].unique())
models = sorted(df["model"].unique())

for task in tasks:
    plt.figure(figsize=(10, 5))
    for model in models:
        g = df[(df["model"] == model) & (df["task"] == task)].sort_values("checkpoint")
        plt.plot(g["checkpoint"], g["score"], marker="o", label=model)
    plt.axhline(THRESHOLD, linestyle="--", linewidth=1, label=f"threshold={THRESHOLD}")
    plt.title(f"ElementalTask-RML: Cross-Model Trajectory — {task}")
    plt.xlabel("Checkpoint")
    plt.ylabel("Score")
    plt.ylim(-0.05, 1.05)
    plt.legend()
    plt.tight_layout()
    safe_task = task.replace(":", "_").replace("/", "_")
    plt.savefig(FIG_DIR / f"05_trajectory_{safe_task}.png", dpi=180)
    plt.show()

## 6. Multi-model stability timeline

For each model, compare its current partial rank ordering against its final rank ordering.

This mirrors Notebook 01, now across model families.

In [ ]:
def partial_rank_stability_by_model(df, threshold=THRESHOLD):
    rows = []
    for model, mdf in df.groupby("model"):
        final = compute_emergence_table(mdf, threshold=threshold)
        final_rank = final.set_index("task")["emergence_rank"]
        for ckpt in sorted(mdf["checkpoint"].unique()):
            partial_df = mdf[mdf["checkpoint"] <= ckpt]
            partial = compute_emergence_table(partial_df, threshold=threshold)
            partial_rank = partial.set_index("task")["emergence_rank"]
            common = final_rank.index.intersection(partial_rank.index)
            if len(common) >= 2:
                if spearmanr is not None:
                    corr = spearmanr(final_rank.loc[common], partial_rank.loc[common]).correlation
                else:
                    corr = final_rank.loc[common].rank().corr(partial_rank.loc[common].rank())
            else:
                corr = np.nan
            rows.append({
                "model": model,
                "checkpoint": ckpt,
                "rank_stability_to_final": corr,
            })
    return pd.DataFrame(rows)

stability = partial_rank_stability_by_model(df)
stability.to_csv(RES_DIR / "05_multi_model_stability.csv", index=False)
stability.head()

In [ ]:
plt.figure(figsize=(12, 6))
for model in models:
    g = stability[stability["model"] == model].sort_values("checkpoint")
    plt.plot(g["checkpoint"], g["rank_stability_to_final"], marker="o", label=model)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(0.7, linestyle=":", linewidth=1, label="monitor threshold")
plt.title("ElementalTask-RML: Multi-Model Rank Stability Timeline")
plt.xlabel("Checkpoint")
plt.ylabel("Spearman correlation to final ordering")
plt.ylim(-1.05, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "05_multi_model_stability.png", dpi=180)
plt.show()

## 7. Cross-model forecasting transfer

Can a simple emergence-order heuristic from one model forecast composite emergence in another?

This is deliberately lightweight and interpretable.

Forecast rule:

> predicted composite checkpoint = source model component mean + target/source scale factor

This is not a full predictive model. It is a minimal test of whether component → composite structure transfers across model families.

In [ ]:
COMPOSITE_MAP = {
    "compositional:copy_then_uppercase": ["simple:copying", "simple:uppercase"],
    "compositional:first_letter_then_uppercase": ["simple:first_letter", "simple:uppercase"],
}

def forecast_transfer_table(emergence, composite_map=COMPOSITE_MAP):
    rows = []
    models = sorted(emergence["model"].unique())
    e = emergence.set_index(["model", "task"])

    # model scale: median atomic emergence checkpoint vs source median atomic emergence checkpoint
    atomic_tasks = sorted(set(sum(composite_map.values(), [])))

    for source in models:
        source_atomic = [
            e.loc[(source, t), "emergence_checkpoint"]
            for t in atomic_tasks
            if (source, t) in e.index and pd.notna(e.loc[(source, t), "emergence_checkpoint"])
        ]
        source_median = np.nanmedian(source_atomic) if source_atomic else np.nan

        for target in models:
            target_atomic = [
                e.loc[(target, t), "emergence_checkpoint"]
                for t in atomic_tasks
                if (target, t) in e.index and pd.notna(e.loc[(target, t), "emergence_checkpoint"])
            ]
            target_median = np.nanmedian(target_atomic) if target_atomic else np.nan
            scale = target_median / source_median if source_median and pd.notna(source_median) else np.nan

            for comp, components in composite_map.items():
                if (target, comp) not in e.index:
                    continue
                source_component_ckpts = []
                for c in components:
                    if (source, c) in e.index:
                        source_component_ckpts.append(e.loc[(source, c), "emergence_checkpoint"])
                if not source_component_ckpts or any(pd.isna(source_component_ckpts)):
                    pred = np.nan
                else:
                    # conservative offset from components to composite learned from source
                    if (source, comp) in e.index:
                        source_comp = e.loc[(source, comp), "emergence_checkpoint"]
                        source_comp_mean = np.nanmean(source_component_ckpts)
                        learned_offset = max(0, source_comp - source_comp_mean) if pd.notna(source_comp) else 0
                    else:
                        learned_offset = 0
                    pred = (np.nanmean(source_component_ckpts) + learned_offset) * scale

                observed = e.loc[(target, comp), "emergence_checkpoint"]
                rows.append({
                    "source_model": source,
                    "target_model": target,
                    "composite_task": comp,
                    "predicted_checkpoint": pred,
                    "observed_checkpoint": observed,
                    "absolute_error": abs(pred - observed) if pd.notna(pred) and pd.notna(observed) else np.nan,
                })
    return pd.DataFrame(rows)

transfer = forecast_transfer_table(emergence)
transfer.to_csv(RES_DIR / "05_forecast_transfer.csv", index=False)
transfer.head(20)

In [ ]:
summary = (
    transfer.dropna(subset=["absolute_error"])
    .groupby(["source_model", "target_model"])["absolute_error"]
    .mean()
    .reset_index(name="mean_absolute_error")
)

pivot_mae = summary.pivot(index="source_model", columns="target_model", values="mean_absolute_error")
pivot_mae.to_csv(RES_DIR / "05_forecast_transfer_mae_matrix.csv")

plt.figure(figsize=(8, 6))
im = plt.imshow(pivot_mae.astype(float))
plt.colorbar(im, label="Mean absolute error")
plt.xticks(range(len(pivot_mae.columns)), pivot_mae.columns, rotation=30, ha="right")
plt.yticks(range(len(pivot_mae.index)), pivot_mae.index)
plt.title("ElementalTask-RML: Cross-Model Forecast Transfer Error")

for i in range(len(pivot_mae.index)):
    for j in range(len(pivot_mae.columns)):
        val = pivot_mae.iloc[i, j]
        if pd.notna(val):
            plt.text(j, i, f"{val:.0f}", ha="center", va="center")

plt.tight_layout()
plt.savefig(FIG_DIR / "05_forecast_transfer_error.png", dpi=180)
plt.show()

pivot_mae

## 8. Generalization summary table

A compact table for README and lab-report use.

In [ ]:
final_stability = (
    stability.sort_values("checkpoint")
    .groupby("model")
    .tail(1)
    .set_index("model")["rank_stability_to_final"]
)

final_scores = (
    emergence.groupby("model")["final_score"]
    .mean()
    .rename("mean_final_score")
)

forecast_mae = (
    transfer[transfer["source_model"] == transfer["target_model"]]
    .groupby("target_model")["absolute_error"]
    .mean()
    .rename("within_model_forecast_mae")
)

summary_table = pd.concat([final_stability, final_scores, forecast_mae], axis=1).reset_index().rename(columns={"index": "model"})
summary_table.to_csv(RES_DIR / "05_generalization_summary.csv", index=False)
summary_table

In [ ]:
plt.figure(figsize=(10, 5))
x = np.arange(len(summary_table))
width = 0.35

plt.bar(x - width/2, summary_table["rank_stability_to_final"], width, label="final rank stability")
plt.bar(x + width/2, summary_table["mean_final_score"], width, label="mean final score")

plt.xticks(x, summary_table["model"], rotation=25, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("ElementalTask-RML: Cross-Model Generalization Summary")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "05_generalization_summary.png", dpi=180)
plt.show()

## 9. Interpretation

ElementalTask suggests partially stable developmental ordering during training.

This notebook explores whether:
- emergence ordering,
- cross-model rank stability,
- and compositional forecasting

generalize across different model families and checkpoint schedules.

Phase I arc:

> observe → interpret → monitor → forecast → compare

Next step: document this arc in `roadmap.md` and avoid notebook sprawl unless Phase II has a specific technical purpose.

In [ ]:
# ---------------------------------------------------------------------
# Export a small Markdown report
# ---------------------------------------------------------------------
report = f'''
# Notebook 05 — Cross-Model Stability

## Purpose

Compare emergence ordering and monitoring behavior across model families.

## Outputs

- `figures/05_cross_model_rank_correlation.png`
- `figures/05_multi_model_stability.png`
- `figures/05_forecast_transfer_error.png`
- `figures/05_generalization_summary.png`
- `results/05_cross_model_emergence.csv`
- `results/05_cross_model_rank_correlation.csv`
- `results/05_multi_model_stability.csv`
- `results/05_forecast_transfer.csv`
- `results/05_generalization_summary.csv`

## Interpretation

This notebook closes the Phase I arc:

observe → interpret → monitor → forecast → compare.

Emergence ≠ magic. Monitor constraints. 📐
'''.strip()

(DOC_DIR / "05_cross_model_stability.md").write_text(report)
print(DOC_DIR / "05_cross_model_stability.md")

## Optional: zip outputs for download

Uncomment and run in Colab to download Notebook 05 outputs.

In [ ]:
# # Optional Colab download block
# import zipfile
# from google.colab import files
#
# EXPORT_NAME = "elemental_task_rml_notebook05_outputs.zip"
# include_paths = [
#     FIG_DIR / "05_cross_model_rank_correlation.png",
#     FIG_DIR / "05_multi_model_stability.png",
#     FIG_DIR / "05_forecast_transfer_error.png",
#     FIG_DIR / "05_generalization_summary.png",
#     RES_DIR / "05_cross_model_emergence.csv",
#     RES_DIR / "05_cross_model_rank_correlation.csv",
#     RES_DIR / "05_multi_model_stability.csv",
#     RES_DIR / "05_forecast_transfer.csv",
#     RES_DIR / "05_generalization_summary.csv",
#     DOC_DIR / "05_cross_model_stability.md",
# ]
#
# with zipfile.ZipFile(EXPORT_NAME, "w", compression=zipfile.ZIP_DEFLATED) as zf:
#     for p in include_paths:
#         if Path(p).exists():
#             zf.write(p, arcname=str(Path(p).relative_to(RML_DIR)))
#
# files.download(EXPORT_NAME)